# RemoveWatermark Colab 版

這份筆記本會在 Google Colab 裡準備環境、安裝專案，並啟動 Web UI（網頁介面）。它支援 LaMa 補圖、模板偵測和 SAM3 偵測。SAM3 需要 CUDA GPU；SAM 3.1 模型約 3.5GB，第一次安裝和下載會比較久。第二格可以把套件下載快取與模型快取放到 Google Drive，之後 runtime 重開時可減少重複下載。LaMa 和模板模式也可在 CPU 執行，但速度會慢很多。

In [ ]:
#@title 1. 取得專案
專案來源 = "Google Drive 資料夾" #@param ["上傳 zip", "GitHub", "Google Drive 資料夾"]
GitHub_網址 = "" #@param {type:"string"}
Google_Drive_專案資料夾 = "/content/drive/MyDrive/RemoveWatermark" #@param {type:"string"}

from pathlib import Path
import os
import shutil
import subprocess
import zipfile

PROJECT_SOURCE_LABEL = 專案來源
GITHUB_URL = GitHub_網址
DRIVE_PROJECT_DIR = Google_Drive_專案資料夾

PROJECT_SOURCE_MAP = {
    "上傳 zip": "upload_zip",
    "GitHub": "github",
    "Google Drive 資料夾": "drive",
}
PROJECT_SOURCE = PROJECT_SOURCE_MAP[PROJECT_SOURCE_LABEL]

WORKDIR = Path("/content/RemoveWatermark")
UPLOAD_ROOT = Path("/content/uploaded_removewatermark")

def print_step(message: str):
    print(f"▶ {message}", flush=True)

def validate_project(path: Path):
    if not (path / "pyproject.toml").exists() or not (path / "src" / "remove_watermark").exists():
        raise FileNotFoundError(f"這不是有效的 RemoveWatermark 專案資料夾：{path}")

def clean_path(path: Path):
    if path.exists():
        print_step(f"清理舊資料：{path}")
        shutil.rmtree(path)

def copy_project(src: Path, dst: Path):
    print_step(f"複製專案：{src} -> {dst}")
    ignore = shutil.ignore_patterns(".git", ".venv", "venv", "env", "output", "tmp", "__pycache__", "*.pyc")
    shutil.copytree(src, dst, ignore=ignore)
    print_step("專案複製完成。")

def extract_project_zip(archive_path: Path, dst: Path):
    dst_root = dst.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            normalized_name = member.filename.replace("\\", "/")
            is_directory = normalized_name.endswith("/") or member.is_dir()
            parts = [part for part in normalized_name.split("/") if part and part != "."]
            if not parts:
                continue
            if any(part == ".." or ":" in part for part in parts):
                raise ValueError(f"zip 內含不安全路徑：{member.filename}")
            target = dst.joinpath(*parts)
            try:
                target.resolve().relative_to(dst_root)
            except ValueError as exc:
                raise ValueError(f"zip 內含不安全路徑：{member.filename}") from exc
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

os.chdir("/content")
clean_path(WORKDIR)

if PROJECT_SOURCE == "github":
    if not GITHUB_URL.strip():
        raise ValueError("請先填 GITHUB_URL。")
    print_step("正在從 GitHub 下載專案。")
    subprocess.run(["git", "clone", GITHUB_URL.strip(), str(WORKDIR)], check=True)
    validate_project(WORKDIR)
    print_step("GitHub 專案下載完成。")
elif PROJECT_SOURCE == "drive":
    from google.colab import drive
    print_step("正在掛載 Google Drive。")
    drive.mount("/content/drive")
    src = Path(DRIVE_PROJECT_DIR)
    if not src.exists():
        raise FileNotFoundError(f"找不到 Google Drive 專案資料夾：{src}")
    validate_project(src)
    copy_project(src, WORKDIR)
else:
    from google.colab import files
    clean_path(UPLOAD_ROOT)
    UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    print("請上傳整個專案的 zip 檔。")
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith(".zip")]
    if not archives:
        raise ValueError("沒有找到 zip 檔。請把專案資料夾壓成 zip 後再上傳。")
    archive_path = Path(archives[0])
    print_step(f"已收到壓縮檔：{archive_path.name}，大小 {archive_path.stat().st_size / (1024 * 1024):.2f} MB。")
    print_step("正在解壓縮專案。")
    extract_project_zip(archive_path, UPLOAD_ROOT)
    print_step("解壓縮完成。")
    print_step("正在尋找專案根目錄。")
    candidates = [
        path.parent
        for path in UPLOAD_ROOT.rglob("pyproject.toml")
        if (path.parent / "src" / "remove_watermark").exists()
    ]
    if not candidates:
        raise FileNotFoundError("zip 裡找不到 RemoveWatermark 專案。")
    print_step(f"找到專案根目錄：{candidates[0]}")
    copy_project(candidates[0], WORKDIR)

os.chdir(WORKDIR)
print(f"✅ 專案已準備好：{WORKDIR}", flush=True)

In [ ]:
#@title 2. 安裝套件
下載_SAM3_模型 = True #@param {type:"boolean"}
使用_Google_Drive_快取 = True #@param {type:"boolean"}
Google_Drive_快取資料夾 = "/content/drive/MyDrive/RemoveWatermark/.cache" #@param {type:"string"}
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

DOWNLOAD_SAM3_MODEL = 下載_SAM3_模型
USE_GOOGLE_DRIVE_CACHE = 使用_Google_Drive_快取
DRIVE_CACHE_DIR = Google_Drive_快取資料夾


def run_step(label: str, command: list[str]):
    print(f"\n▶ {label}", flush=True)
    subprocess.run(command, check=True)
    print(f"✓ {label} 完成", flush=True)


def pip_install(label: str, *packages):
    run_step(label, [sys.executable, "-m", "pip", "install", *packages])


WORKDIR = Path("/content/RemoveWatermark")
SRC_DIR = WORKDIR / "src"


def ensure_project_ready():
    if not (WORKDIR / "pyproject.toml").exists() or not (SRC_DIR / "remove_watermark").exists():
        raise RuntimeError("找不到 RemoveWatermark 專案。請先執行第 1 格取得專案，再重新執行這格。")
    os.chdir(WORKDIR)
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
    print(f"✓ 使用專案資料夾：{WORKDIR}", flush=True)


def setup_drive_cache():
    if not USE_GOOGLE_DRIVE_CACHE:
        print("✓ 已關閉 Google Drive 快取。", flush=True)
        return
    from google.colab import drive
    print("\n▶ 掛載 Google Drive 並準備快取資料夾", flush=True)
    drive.mount("/content/drive")
    cache_root = Path(DRIVE_CACHE_DIR).expanduser()
    cache_dirs = {
        "PIP_CACHE_DIR": cache_root / "pip",
        "MODELSCOPE_CACHE": cache_root / "modelscope",
        "HF_HOME": cache_root / "huggingface",
        "TORCH_HOME": cache_root / "torch",
    }
    for name, path in cache_dirs.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[name] = str(path)
        print(f"{name} = {path}", flush=True)
    print("✓ Google Drive 快取已啟用；runtime 重開後仍需安裝，但會盡量重用下載檔。", flush=True)


def verify_numpy_binary():
    try:
        import numpy as np
        import numpy.random  # noqa: F401
    except ValueError as exc:
        raise RuntimeError(
            "NumPy 二進位套件版本不一致。這通常是某個套件在同一個 runtime 裡替換過 NumPy。"
            "請到『執行階段 -> 重新啟動工作階段』，再從第 1 格開始重新執行。"
        ) from exc
    print(f"✓ NumPy 檢查完成：{np.__version__}", flush=True)
    return f"numpy=={np.__version__}"


ensure_project_ready()
setup_drive_cache()

pip_install("更新 Python 安裝工具", "--upgrade", "pip", "setuptools<81", "wheel")
NUMPY_REQUIREMENT = verify_numpy_binary()
print(f"✓ 保留 Colab 目前 NumPy 版本：{NUMPY_REQUIREMENT}", flush=True)

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    pip_install("安裝 PyTorch", "torch", "torchvision", NUMPY_REQUIREMENT)
else:
    print("✓ PyTorch 已存在，略過重新安裝。", flush=True)

pip_install("安裝基礎影像處理套件", NUMPY_REQUIREMENT, "opencv-python==4.11.0.86", "Pillow>=9.5,<11.0", "fire==0.5.0", "six", "termcolor")
pip_install("安裝 LaMa 補圖套件", "simple-lama-inpainting==0.1.2", "--no-deps")
if DOWNLOAD_SAM3_MODEL:
    pip_install("安裝 SAM3/AI 支援套件", NUMPY_REQUIREMENT, "timm>=1.0.17", "tqdm", "ftfy==6.1.1", "regex", "iopath>=0.1.10", "typing_extensions", "huggingface_hub", "einops>=0.8", "modelscope>=1.29", "psutil>=5.9", "pycocotools>=2.0.10", "setuptools<81")
    pip_install("安裝 SAM3 套件", "sam3 @ git+https://github.com/facebookresearch/sam3.git", "--no-deps")
else:
    print("✓ 已略過 SAM3/AI 偵測套件安裝。", flush=True)
verify_numpy_binary()
pip_install("安裝 RemoveWatermark 專案", "-e", ".", "--no-deps")

if DOWNLOAD_SAM3_MODEL:
    run_step("下載 SAM 3.1 模型，約 3.5GB，第一次會比較久", [sys.executable, "-m", "remove_watermark", "--download-sam3-model"])
else:
    print("已略過 SAM3 套件安裝和模型下載；要用 SAM3 前請回來開啟『下載_SAM3_模型』。")

import torch
print(f"PyTorch 版本：{torch.__version__}")
print(f"CUDA GPU 可用：{torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU：{torch.cuda.get_device_name(0)}")

In [ ]:
#@title 3. 快速測試 LaMa 和 SAM3 環境
import os
import sys
from pathlib import Path
import numpy as np
import numpy.random  # noqa: F401
import torch

WORKDIR = Path("/content/RemoveWatermark")
SRC_DIR = WORKDIR / "src"
if not (SRC_DIR / "remove_watermark").exists():
    raise RuntimeError("找不到 RemoveWatermark 專案套件。請先執行第 1 格取得專案，再執行第 2 格安裝套件。")
os.chdir(WORKDIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from remove_watermark.core import build_simple_lama_runner

device = "cuda" if torch.cuda.is_available() else "cpu"
runner = build_simple_lama_runner(device)
image = np.full((96, 96, 3), 220, dtype=np.uint8)
image[32:64, 32:64] = [30, 30, 30]
mask = np.zeros((96, 96), dtype=np.uint8)
mask[32:64, 32:64] = 255
result = runner(image, mask)
print(f"LaMa 測試完成：{result.shape}, {result.dtype}, device={device}")

print(f"CUDA GPU 可用：{torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU：{torch.cuda.get_device_name(0)}")

if globals().get("DOWNLOAD_SAM3_MODEL", False):
    try:
        import modelscope  # noqa: F401
        from sam3.model.sam3_image_processor import Sam3Processor  # noqa: F401
        from sam3.model_builder import build_sam3_image_model  # noqa: F401
    except Exception as exc:
        raise RuntimeError("SAM3 套件檢查失敗。請先執行第 2 格安裝套件。") from exc

    print("SAM3 套件檢查完成：modelscope、Sam3Processor、build_sam3_image_model 可用。")
    if not torch.cuda.is_available():
        print("提醒：目前沒有 CUDA GPU，SAM3 偵測會無法使用；Web UI 請切到模板模式。")
    else:
        print("SAM3 需要的 CUDA GPU 可用。")
else:
    print("已略過 SAM3 套件檢查；要用 SAM3 請回到第 2 格開啟『下載_SAM3_模型』。")

In [ ]:
#@title 4. 啟動 Web UI
import os
import subprocess
import sys
import time
from pathlib import Path
import torch
from google.colab import output

PORT = 8765
WORKDIR = Path("/content/RemoveWatermark")
SRC_DIR = WORKDIR / "src"
if not (SRC_DIR / "remove_watermark").exists():
    raise RuntimeError("找不到 RemoveWatermark 專案套件。請先執行第 1 格取得專案，再執行第 2 格安裝套件。")
os.chdir(WORKDIR)
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")

if globals().get("USE_GOOGLE_DRIVE_CACHE", False):
    settings_file = Path(globals().get("DRIVE_CACHE_DIR", "/content/drive/MyDrive/RemoveWatermark/.cache")).expanduser().parent / "web_advanced_settings.json"
else:
    settings_file = WORKDIR / "output" / ".editor_state" / "advanced_settings.json"
settings_file.parent.mkdir(parents=True, exist_ok=True)

try:
    ui_process.terminate()
    ui_process.wait(timeout=5)
except Exception:
    pass

ui_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "remove_watermark.web",
        "--host",
        "0.0.0.0",
        "--port",
        str(PORT),
        "--settings-file",
        str(settings_file),
        "--no-open-browser",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(WORKDIR),
    env=env,
)
time.sleep(3)
if ui_process.poll() is not None:
    print(ui_process.stdout.read())
    raise RuntimeError("Web UI 啟動失敗。")

print(f"Web UI 已啟動，連接埠：{PORT}")
print(f"進階設定會保存到：{settings_file}")
sam3_ready = bool(globals().get("DOWNLOAD_SAM3_MODEL", False))
if sam3_ready and torch.cuda.is_available():
    print("可以使用 SAM3 偵測模式；第一次載入模型可能會比較久。")
elif not sam3_ready:
    print("這次略過 SAM3 準備；請用模板模式，或回第 2 格開啟『下載_SAM3_模型』後重新安裝。")
else:
    print("目前沒有 CUDA GPU，SAM3 偵測模式不可用；請在 Web UI 切到模板模式。")
try:
    from google.colab.output import eval_js
    print("如果下方內嵌視窗沒有出現，請開這個備用連結：")
    print(eval_js(f"google.colab.kernel.proxyPort({PORT})"))
except Exception as exc:
    print(f"無法產生備用連結：{exc}")
output.serve_kernel_port_as_iframe(PORT, height=800)

## 批量命令列模式（可選）

如果不想用網頁介面，也可以把圖片放到 `/content/RemoveWatermark/input`，再執行下面這格。「偵測模式」選 `SAM3` 時不需要模板；選「模板」時才會使用 `/content/RemoveWatermark/templates`。SAM3 需要 CUDA GPU；如果第 2 格略過 SAM3 準備，批量處理會自動改用模板模式。

In [ ]:
#@title 5. 批量處理（可選）
偵測模式 = "SAM3" #@param ["SAM3", "模板"]
輸入資料夾 = "/content/RemoveWatermark/input" #@param {type:"string"}
模板資料夾 = "/content/RemoveWatermark/templates" #@param {type:"string"}
輸出資料夾 = "/content/RemoveWatermark/output" #@param {type:"string"}
SAM3_提示 = "watermark. text watermark. transparent watermark. logo. stamp." #@param {type:"string"}
SAM3_信心門檻 = 0.20 #@param {type:"number"}
SAM3_最多區域 = 60 #@param {type:"integer"}
import os
import subprocess
import sys
from pathlib import Path

DETECTOR_LABEL = 偵測模式
INPUT_DIR = 輸入資料夾
TEMPLATE_DIR = 模板資料夾
OUTPUT_DIR = 輸出資料夾
AI_PROMPT = SAM3_提示
AI_BOX_THRESHOLD = SAM3_信心門檻
AI_MAX_DETECTIONS = SAM3_最多區域

DETECTOR_MAP = {
    "SAM3": "sam3",
    "模板": "template",
}
DETECTOR = DETECTOR_MAP[DETECTOR_LABEL]

WORKDIR = Path("/content/RemoveWatermark")
SRC_DIR = WORKDIR / "src"
if not (SRC_DIR / "remove_watermark").exists():
    raise RuntimeError("找不到 RemoveWatermark 專案套件。請先執行第 1 格取得專案，再執行第 2 格安裝套件。")
os.chdir(WORKDIR)
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")

cmd = [
    sys.executable,
    "-m",
    "remove_watermark",
    "--input",
    INPUT_DIR,
    "--output",
    OUTPUT_DIR,
    "--lama-device",
    "auto",
]

active_detector = DETECTOR
if active_detector == "sam3" and not globals().get("DOWNLOAD_SAM3_MODEL", False):
    print("這次未準備 SAM3，批量處理自動改用模板模式。")
    active_detector = "template"

cmd.extend(["--detector", active_detector])

if active_detector == "template":
    cmd.extend(["--template", TEMPLATE_DIR])
else:
    cmd.extend([
        "--ai-prompt",
        AI_PROMPT,
        "--ai-box-threshold",
        str(AI_BOX_THRESHOLD),
        "--ai-max-detections",
        str(AI_MAX_DETECTIONS),
    ])

subprocess.run(cmd, check=True, cwd=str(WORKDIR), env=env)
print(f"完成，結果在：{OUTPUT_DIR}")